# 1. Microstructure Research

流程: Orderbook 散点图 -> Wall Mid -> Normalized 图 -> Return ACF -> 交易分类 -> Adverse Selection

In [63]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices, load_prices_wide, load_trades
from utils.orderbook import compute_wall_mid, compute_spread, return_autocorrelation, adverse_selection, order_interval_distribution, trade_interval_distribution, normalized_level_trade_stats
from utils.viz import plot_orderbook_scatter, plot_autocorrelation, plot_trade_profile, plot_normalized_orderbook, plot_interval_distribution
import polars as pl
import numpy as np

In [76]:
# === 配置 ===
ROUND = 0
DAY = -1
PRODUCT = 'EMERALDS'  # 改成你要分析的产品

# 逆向工程过滤开关（按量过滤）
FILTER_BY_VOLUME = False
MIN_ORDER_VOLUME = 10
MIN_TRADE_QUANTITY = 3

## Step 1: 加载数据

In [77]:
prices_long = load_prices(ROUND, DAY)
prices_wide = load_prices_wide(ROUND, DAY)
trades = load_trades(ROUND, DAY)

print('Products:', prices_long['product'].unique().to_list())
print('Prices shape:', prices_long.shape)
print('Trades shape:', trades.shape)
trades.head()

Products: ['TOMATOES', 'EMERALDS']
Prices shape: (81043, 8)
Trades shape: (631, 7)


timestamp,buyer,seller,product,currency,price,quantity
i64,str,str,str,str,f64,i64
3200,null,null,"""EMERALDS""","""XIRECS""",9992.0,8
3400,null,null,"""TOMATOES""","""XIRECS""",5009.0,2
5000,null,null,"""EMERALDS""","""XIRECS""",9992.0,7
7000,null,null,"""TOMATOES""","""XIRECS""",5010.0,4
9600,null,null,"""TOMATOES""","""XIRECS""",4999.0,5


## Step 2: Orderbook 散点图（核心可视化）

In [78]:
# 计算 wall mid
wall_mid_df = compute_wall_mid(prices_wide.filter(pl.col('product') == PRODUCT))
wall_mid_df.head()

day,timestamp,product,bid_wall_price,bid_wall_volume,ask_wall_price,ask_wall_volume,wall_mid,raw_mid
i64,i64,str,i64,i64,i64,i64,f64,f64
-1,0,"""EMERALDS""",9990,29,10010,29,10000.0,10000.0
-1,100,"""EMERALDS""",9990,22,10010,22,10000.0,10000.0
-1,200,"""EMERALDS""",9990,20,10010,20,10000.0,10000.0
-1,300,"""EMERALDS""",9990,29,10010,29,10000.0,10000.0
-1,400,"""EMERALDS""",9990,25,10010,25,10000.0,10000.0


In [79]:
# Orderbook 散点图 + wall mid + trades
fig = plot_orderbook_scatter(
    prices_long, trades, product=PRODUCT, day=DAY, wall_mid_df=wall_mid_df,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
)
fig.show()

## Step 3: Normalized Orderbook（去趋势）

In [68]:
fig = plot_normalized_orderbook(
    prices_long, wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig.show()

## Step 4: Spread 统计

In [69]:
spread_df = compute_spread(prices_wide.filter(pl.col('product') == PRODUCT))
print('Wall spread stats:')
print(spread_df['wall_spread'].describe())
print(f"\nWall mid vs Raw mid diff: {(spread_df['wall_mid'] - spread_df['raw_mid']).mean():.4f}")

Wall spread stats:
shape: (9, 2)
┌────────────┬─────────┐
│ statistic  ┆ value   │
│ ---        ┆ ---     │
│ str        ┆ f64     │
╞════════════╪═════════╡
│ count      ┆ 10000.0 │
│ null_count ┆ 0.0     │
│ mean       ┆ 20.0    │
│ std        ┆ 0.0     │
│ min        ┆ 20.0    │
│ 25%        ┆ 20.0    │
│ 50%        ┆ 20.0    │
│ 75%        ┆ 20.0    │
│ max        ┆ 20.0    │
└────────────┴─────────┘

Wall mid vs Raw mid diff: 0.0028


## Step 5: Return Autocorrelation

In [70]:
wm = wall_mid_df.sort('timestamp')['wall_mid'].to_numpy()
acf, ci = return_autocorrelation(wm, max_lag=20)

fig = plot_autocorrelation(acf, ci, title=f'{PRODUCT} Return ACF')
fig.show()

# 解读
print(f'ACF(1) = {acf[0]:.4f}, 95% CI = +/-{ci:.4f}')
if abs(acf[0]) < ci:
    print('-> 不可预测，专注做市')
elif acf[0] < -ci:
    print('-> 短期均值回复，可以逆向交易')
else:
    print('-> 短期动量，taker 有 informed signal')

/Users/leoliu/.virtualenvs/sandbox/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning:

invalid value encountered in divide



ACF(1) = nan, 95% CI = +/-0.0196
-> 短期动量，taker 有 informed signal


## Step 6: 交易者分类

In [71]:
product_trades = trades.filter(pl.col('product') == PRODUCT)

if 'buyer' in trades.columns:
    fig = plot_trade_profile(trades, product=PRODUCT)
    fig.show()
    
    # 每个 trader 的行为模式
    for trader in product_trades['buyer'].unique().to_list():
        if not trader:
            continue
        tt = product_trades.filter(pl.col('buyer') == trader)
        print(f"\n{trader}: {len(tt)} buys, qty mode={tt['quantity'].mode().to_list()}, "
              f"avg_price={tt['price'].mean():.1f}")
else:
    print('No buyer/seller info in this round')

## Step 7: 订单间隔 / 成交间隔分布

In [72]:
order_intervals = order_interval_distribution(
    prices_long,
    product=PRODUCT,
    day=DAY,
    min_order_volume=MIN_ORDER_VOLUME if FILTER_BY_VOLUME else None,
)
trade_intervals = trade_interval_distribution(
    trades,
    product=PRODUCT,
    day=DAY,
    min_trade_quantity=MIN_TRADE_QUANTITY if FILTER_BY_VOLUME else None,
)

print(f"Order intervals count: {order_intervals.height}, mean={order_intervals['interval_ms'].mean() if order_intervals.height else 'NA'}")
print(f"Trade intervals count: {trade_intervals.height}, mean={trade_intervals['interval_ms'].mean() if trade_intervals.height else 'NA'}")

wm_overlay = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_overlay.columns:
    wm_overlay = wm_overlay.filter(pl.col('day') == DAY)

fig_o = plot_interval_distribution(
    order_intervals,
    title=f"Order Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_o.show()

fig_t = plot_interval_distribution(
    trade_intervals,
    title=f"Trade Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_t.show()

Order intervals count: 9999, mean=100.0
Trade intervals count: 190, mean=5078.9473684210525


## Step 8: Wall Mid 奇偶统计

In [73]:
wm_stats = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_stats.columns:
    wm_stats = wm_stats.filter(pl.col('day') == DAY)
wm_stats = wm_stats.filter(pl.col('wall_mid').is_not_null()).with_columns(
    (2*pl.col('wall_mid') % 2 == 0).alias('is_even')
)

ratio_tbl = wm_stats.group_by('is_even').agg(pl.len().alias('n')).with_columns(
    (pl.col('n') / pl.col('n').sum()).alias('ratio')
).sort('is_even')

print('2 * Wall mid 奇偶占比:')
print(ratio_tbl)

trades_with_wm = (
    trades.filter(pl.col('product') == PRODUCT)
    .join(wm_stats.select(['timestamp', 'product', 'wall_mid', 'is_even']), on=['timestamp', 'product'], how='left')
    .filter(pl.col('is_even').is_not_null())
)

qty_col = 'quantity' if 'quantity' in trades_with_wm.columns else ('volume' if 'volume' in trades_with_wm.columns else None)
if qty_col is None:
    print('Trades 中没有 quantity/volume 列，无法统计成交量。')
else:
    trade_qty_tbl = (
        trades_with_wm.group_by('is_even')
        .agg([
            pl.col(qty_col).abs().sum().alias('total_trade_qty'),
            pl.len().alias('trade_count'),
        ])
        .sort('is_even')
    )
    print('\nWall mid 奇偶时的总成交量:')
    print(trade_qty_tbl)

even_ratio = ratio_tbl.filter(pl.col('is_even') == True)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == True).height else 0.0
odd_ratio = ratio_tbl.filter(pl.col('is_even') == False)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == False).height else 0.0
print(f"\nEven ratio={even_ratio:.4f}, Odd ratio={odd_ratio:.4f}, Δfrom 50%={abs(even_ratio-0.5):.4f}")

2 * Wall mid 奇偶占比:
shape: (1, 3)
┌─────────┬───────┬───────┐
│ is_even ┆ n     ┆ ratio │
│ ---     ┆ ---   ┆ ---   │
│ bool    ┆ u32   ┆ f64   │
╞═════════╪═══════╪═══════╡
│ true    ┆ 10000 ┆ 1.0   │
└─────────┴───────┴───────┘

Wall mid 奇偶时的总成交量:
shape: (1, 3)
┌─────────┬─────────────────┬─────────────┐
│ is_even ┆ total_trade_qty ┆ trade_count │
│ ---     ┆ ---             ┆ ---         │
│ bool    ┆ i64             ┆ u32         │
╞═════════╪═════════════════╪═════════════╡
│ true    ┆ 1040            ┆ 191         │
└─────────┴─────────────────┴─────────────┘

Even ratio=1.0000, Odd ratio=0.0000, Δfrom 50%=0.5000


## Step 9: 标准化档位成交统计

In [74]:
level_stats = normalized_level_trade_stats(
    trades=trades,
    wall_mid_df=wall_mid_df,
    product=PRODUCT,
    day=DAY,
    round_to=None,
)

print('标准化档位成交统计（norm_level = trade_price - wall_mid）:')
print(level_stats)

if level_stats.height:
    print('\n按成交次数排序 Top10:')
    print(level_stats.sort('trade_count', descending=True).head(10))

# 验证：norm_level 是否符合 0.5 网格，以及 .0/.5 占比
norm = level_stats.select(['norm_level', 'trade_count']).with_columns([
    (pl.col('norm_level') * 2).round(8).alias('norm_x2'),
]).with_columns([
    ((pl.col('norm_x2') - pl.col('norm_x2').round(0)).abs() < 1e-8).alias('is_half_grid'),
    (pl.col('norm_x2').round(0) % 2 == 0).alias('is_integer_level'),
])

total_trades = norm['trade_count'].sum() if norm.height else 0
on_grid_trades = norm.filter(pl.col('is_half_grid'))['trade_count'].sum() if norm.height else 0
int_trades = norm.filter(pl.col('is_half_grid') & pl.col('is_integer_level'))['trade_count'].sum() if norm.height else 0
half_trades = norm.filter(pl.col('is_half_grid') & (~pl.col('is_integer_level')))['trade_count'].sum() if norm.height else 0

print(f"\nHalf-grid integrity: {on_grid_trades}/{total_trades} = {on_grid_trades/total_trades if total_trades else 0:.4f}")
print(f"Integer-level trades (.0): {int_trades} ({int_trades/total_trades if total_trades else 0:.4f})")
print(f"Half-level trades (.5): {half_trades} ({half_trades/total_trades if total_trades else 0:.4f})")

标准化档位成交统计（norm_level = trade_price - wall_mid）:
shape: (3, 4)
┌────────────┬─────────────┬────────────────┬──────────────┐
│ norm_level ┆ trade_count ┆ total_quantity ┆ avg_quantity │
│ ---        ┆ ---         ┆ ---            ┆ ---          │
│ f64        ┆ u32         ┆ i64            ┆ f64          │
╞════════════╪═════════════╪════════════════╪══════════════╡
│ -8.0       ┆ 98          ┆ 530            ┆ 5.408163     │
│ 0.0        ┆ 3           ┆ 14             ┆ 4.666667     │
│ 8.0        ┆ 90          ┆ 496            ┆ 5.511111     │
└────────────┴─────────────┴────────────────┴──────────────┘

按成交次数排序 Top10:
shape: (3, 4)
┌────────────┬─────────────┬────────────────┬──────────────┐
│ norm_level ┆ trade_count ┆ total_quantity ┆ avg_quantity │
│ ---        ┆ ---         ┆ ---            ┆ ---          │
│ f64        ┆ u32         ┆ i64            ┆ f64          │
╞════════════╪═════════════╪════════════════╪══════════════╡
│ -8.0       ┆ 98          ┆ 530            ┆ 5.408163

## Step 10: Adverse Selection 检验

In [75]:
wm_series = wall_mid_df.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid'])
as_result = adverse_selection(product_trades, wm_series)

print('Adverse Selection (taker PnL after trade):')
for h, v in as_result.items():
    label = 'SAFE (taker=noise)' if v <= 0 else 'DANGER (taker=informed)'
    print(f'  h={h:3d} steps: {v:+.4f}  {label}')

Adverse Selection (taker PnL after trade):
  h=  1 steps: +0.0000  SAFE (taker=noise)
  h=  5 steps: +0.0000  SAFE (taker=noise)
  h= 10 steps: +0.0000  SAFE (taker=noise)
  h= 50 steps: +0.0000  SAFE (taker=noise)


## 结论

在这里写你的观察...